<a href="https://colab.research.google.com/github/anthony-yt/deepwater-image-segmentation-classical-quantum/blob/feat%2Fsam-mask-audit/notebooks/2_3_sam_mask_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 4.3 - Auditoria de las mascaras SAM de 3 puntos

**Objetivo.** Cuantificar que proporcion de las 105 mascaras generadas con prompts de 3 puntos
(`Dataset_Processed/SAM_Masks_Frames_CLAHE`) es utilizable como etiqueta de entrenamiento, para
decidir si sirven o deben sustituirse por mascaras curadas.

**Por que importa.** Esas 105 mascaras son las que `mmseg_pipes` usa como anotacion, es decir, son
las etiquetas con las que se entreno el U-Net de la Task 4.2. Si no son utilizables, ningun
resultado medido sobre ellas lo es.

**Orden de trabajo.** El criterio de clasificacion se fija en la seccion 1, *antes* de mirar
cualquier mascara. Las secciones 2 y 3 son medicion automatica. La clasificacion visual
(seccion 5) se hace despues, aplicando el criterio ya escrito.

## 1. Criterio de clasificacion - fijado antes de mirar las mascaras

Una mascara se clasifica en una sola de estas tres categorias.

### `aceptable`
Sirve como etiqueta de entrenamiento sin retoques. Debe cumplir **todas**:
1. La estructura visible en el frame esta cubierta en su mayor parte.
2. No marca como estructura regiones extensas de agua o lecho marino.
3. La forma de la mascara es reconocible como el objeto (casco, reja, helice, tuberia), no una
   mancha difusa.
4. El borde sigue el contorno del objeto con un error del orden de unos pocos pixeles.

### `parcial`
Contiene informacion correcta pero no es usable sin corregirla. Al menos una de:
1. Cubre solo una parte de la estructura y omite el resto.
2. Marca la estructura correcta mas alguna region de fondo adyacente.
3. La forma es correcta pero el borde esta claramente desplazado o dilatado.
4. Esta fragmentada en varios trozos que, juntos, si corresponden a la estructura.

### `mala`
No contiene informacion aprovechable. Al menos una de:
1. Marca practicamente todo el frame como estructura (o practicamente nada).
2. Marca una region que no corresponde a ninguna estructura.
3. La mascara es ruido: muchas componentes dispersas sin relacion con el objeto.
4. Esta invertida respecto a lo que es estructura y lo que es fondo.

### Reglas de decision
- La clasificacion es **visual**, mirando el overlay. Los indicadores numericos de la seccion 3
  (fraccion de estructura, numero de componentes) sirven para priorizar y para justificar por
  escrito, pero no deciden por si solos.
- Caso de duda entre dos categorias: se asigna la **peor** de las dos y se anota el motivo en
  la columna `nota`.
- La definicion de estructura es la del protocolo 5.0 de Anthony: todo objeto solido de origen
  humano; agua y lecho marino son fondo; cables, objetos sueltos y sombras se asignan a fondo.

### Muestreo
- **Semilla registrada: `20261101`** (`numpy.random.default_rng(20261101)`).
- Muestra **estratificada por video**: 8 frames de Video3, 6 de Video6, 4 de Video7 = **18 frames**,
  dentro del rango de 15 a 20 que pide la tarea. La estratificacion es aproximadamente proporcional
  al numero de mascaras por video (49 / 24 / 10) con un minimo de 4.
- Los videos 1, 2, 4 y 5 quedan fuera: Video1 ya se analizo en la Task 2.1 y los videos 2, 4 y 5 no
  tienen mascaras SAM.

## 2. Entorno y datos

In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage

REPO_URL = "https://github.com/anthony-yt/deepwater-image-segmentation-classical-quantum.git"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Under_Water_Dataset")
else:
    DRIVE_ROOT = Path(os.environ.get("DW_DATA_ROOT", "Under_Water_Dataset")).resolve()

PROCESSED = DRIVE_ROOT / "Dataset_Processed"
FRAMES = PROCESSED / "Frames_CLAHE"
SAM_MASKS = PROCESSED / "SAM_Masks_Frames_CLAHE"
CURATED = PROCESSED / "Test_Set_Curado"
# Los notebooks no escriben en el repositorio: la salida va a Drive.
AUDIT_CSV = PROCESSED / "Registro_Etiquetado" / "auditoria_4_3_Ariana.csv"
AUDIT_CSV.parent.mkdir(parents=True, exist_ok=True)

SEED = 20261101
STRATA = {3: 8, 6: 6, 7: 4}

for name, d in [("frames CLAHE", FRAMES), ("mascaras SAM", SAM_MASKS), ("curadas", CURATED)]:
    n = len(list(d.glob("*.png"))) + len(list(d.glob("*.jpg")))
    print(f"{name:16s} {n:4d}  {d}")

In [ ]:
def load_mask(path) -> np.ndarray:
    """Mascara booleana. convert("L") aplica la paleta, asi que sirve para PNG P, L y RGB.

    Hace falta: en Test_Set_Curado hay 19 archivos en modo P, 8 en L y 1 en RGB, y 3 no son
    estrictamente binarios. Leerlos con cv2.IMREAD_GRAYSCALE devuelve el indice de paleta, no el
    color, y en los de paleta invertida sale la mascara al reves.
    """
    grey = np.asarray(Image.open(path).convert("L"))
    return grey > 0 if grey.max() <= 1 else grey > 127


def load_frame(frame_id: str):
    for ext in (".jpg", ".jpeg", ".png"):
        p = FRAMES / f"{frame_id}{ext}"
        if p.exists():
            return np.asarray(Image.open(p).convert("RGB"))
    return None


def video_of(name: str) -> int:
    m = re.search(r"Video(\d+)", name)
    return int(m.group(1)) if m else -1


def components(mask: np.ndarray) -> int:
    return int(ndimage.label(mask)[1])


def metrics(pred: np.ndarray, gt: np.ndarray) -> dict:
    tp = int((pred & gt).sum())
    fp = int((pred & ~gt).sum())
    fn = int((~pred & gt).sum())
    union = tp + fp + fn
    return {"iou": 1.0 if union == 0 else tp / union,
            "dice": 1.0 if union == 0 else 2 * tp / (2 * tp + fp + fn),
            "precision": tp / (tp + fp) if tp + fp else np.nan,
            "recall": tp / (tp + fn) if tp + fn else np.nan}


sam_paths = {p.stem.replace("_mask", ""): p for p in sorted(SAM_MASKS.glob("*.png"))}
curated_paths = {p.stem.replace("_mask", ""): p for p in sorted(CURATED.glob("*.png"))}
print(f"{len(sam_paths)} mascaras SAM | {len(curated_paths)} curadas | "
      f"{len(set(sam_paths) & set(curated_paths))} frames con ambas")

## 3. Medicion automatica del universo completo

Antes de mirar nada, se mide todo. Dos indicadores por mascara: la fraccion de pixeles marcados
como estructura y el numero de componentes conexas, como proxy de fragmentacion.

In [ ]:
rows = []
for fid, path in sam_paths.items():
    mask = load_mask(path)
    rows.append({"video": video_of(fid), "frame_id": fid, "fg": float(mask.mean()),
                 "components": components(mask), "height": mask.shape[0], "width": mask.shape[1],
                 "has_gt": fid in curated_paths})
universe = pd.DataFrame(rows).sort_values(["video", "frame_id"]).reset_index(drop=True)

summary = universe.groupby("video").agg(
    n=("fg", "size"), fg_mean=("fg", "mean"), fg_sd=("fg", "std"),
    fg_min=("fg", "min"), fg_max=("fg", "max"),
    comp_median=("components", "median"), comp_max=("components", "max"),
    con_curado=("has_gt", "sum"))
summary["base_iou_si_todo_estructura"] = summary["fg_mean"]
print(summary.round(4).to_string())

sizes = universe.apply(lambda r: f"{r.width}x{r.height}", axis=1).value_counts().to_dict()
print(f"\ntotal: {len(universe)} mascaras, tamanos: {sizes}")

for label, subset in [("las 105 mascaras", universe),
                      ("solo videos 3, 6, 7 (el universo a auditar)",
                       universe[universe.video.isin(STRATA)])]:
    print(f"\n--- {label}: {len(subset)} mascaras ---")
    print(f"  fg > 0.90 (casi todo estructura): {(subset.fg > 0.90).sum()}")
    print(f"  fg < 0.05 (casi vacias):          {(subset.fg < 0.05).sum()}")
    print(f"  componentes > 50 (fragmentadas):  {(subset.components > 50).sum()}")

### 3.1 Contra el ground truth, donde existe

La Task 5.1 termino el 2026-11-10: hay **70 mascaras curadas**, 22 de Video1 y 8 de cada uno de
Video2 a Video7. Las mascaras SAM existen para los videos 1, 3, 6 y 7, asi que el solape donde la
calidad se puede *medir* en vez de juzgarse a ojo sube de los 28 frames de noviembre a **46**, y
ahora cubre cuatro videos en lugar de dos.

Se incluye la linea base "todo estructura" porque sin ella un IoU no se puede interpretar: en
Video6 la estructura real ocupa el 8 % del frame y en Video2 el 93 %.

In [ ]:
rows = []
for fid in sorted(set(sam_paths) & set(curated_paths)):
    gt, pred = load_mask(curated_paths[fid]), load_mask(sam_paths[fid])
    if gt.shape != pred.shape:
        print(f"descartado por tamano distinto: {fid} {gt.shape} vs {pred.shape}")
        continue
    row = {"video": video_of(fid), "frame_id": fid,
           "fg_gt": float(gt.mean()), "fg_sam": float(pred.mean()),
           "components": components(pred)}
    row.update(metrics(pred, gt))
    base = metrics(np.ones_like(gt), gt)
    row["base_iou"] = base["iou"]
    row["gain_over_base"] = row["iou"] - base["iou"]
    rows.append(row)
versus_gt = pd.DataFrame(rows)

per_video = versus_gt.groupby("video").agg(
    n=("iou", "size"), fg_gt=("fg_gt", "mean"), fg_sam=("fg_sam", "mean"),
    iou=("iou", "mean"), dice=("dice", "mean"),
    precision=("precision", "mean"), recall=("recall", "mean"),
    base_iou=("base_iou", "mean"), gain=("gain_over_base", "mean"),
    beats_base=("gain_over_base", lambda s: int((s > 0).sum())),
    comp_median=("components", "median"))
print(per_video.round(4).to_string())

In [ ]:
print("Video6 - la unica tuberia real del dataset - frame por frame:\n")
cols = ["frame_id", "fg_gt", "fg_sam", "iou", "precision", "recall", "components"]
print(versus_gt[versus_gt.video == 6][cols].round(4).to_string(index=False))

### 3.2 Coherencia con la Task 2.1

El notebook `2_1_ground_truth_reconciliation.ipynb` midio estas mismas mascaras contra las curadas
de Video1 y obtuvo IoU 0,664 y linea base 0,718. La celda siguiente comprueba que esta auditoria
reproduce esos numeros; si no lo hace, algo cambio en los datos y el resto del notebook no es
comparable con el sprint anterior.

In [ ]:
v1 = versus_gt[versus_gt.video == 1]
print(f"Video1: {len(v1)} frames con curado (la Task 2.1 midio 20)")
print(f"IoU de v3 sobre Video1     : {v1.iou.mean():.4f}   (Task 2.1: 0.664)")
print(f"Linea base todo estructura : {v1.base_iou.mean():.4f}   (Task 2.1: 0.718)")

# El 2026-11-12 Video1 paso de 20 a 22 mascaras curadas, asi que las medias ya no son identicas
# a las de la Task 2.1: con 22 frames dan 0.6305 y 0.7409. Lo que tiene que seguir cumpliendose
# es el rango, y sobre todo que la paleta se lea bien: 6 de las mascaras en modo P tienen la
# paleta invertida, y leer el indice en vez del color hunde la linea base muy por debajo de 0,7.
assert len(v1) in (20, 22), f"Video1 tiene {len(v1)} curadas; revisa la tabla de arriba"
assert 0.60 <= v1.iou.mean() <= 0.70, "IoU de Video1 fuera del rango conocido"
assert 0.70 <= v1.base_iou.mean() <= 0.75, "linea base sospechosa: paleta mal leida?"
print("\ncoherente con la Task 2.1, con 2 mascaras curadas mas en Video1")

### 3.3 Dos modos de fallo, no uno

Con solo Video1 y Video6 parecia que el prompt fijo producia siempre mascaras que lo pintan todo.
Con Video3 y Video7 medidos se ve que hay **dos** fallos distintos, y la celda siguiente los
separa: SAM sobresegmenta cuando los tres puntos caen sobre el fondo, y subsegmenta cuando caen
sobre una parte de la estructura y no sobre el resto. La diferencia importa porque una mascara que
subsegmenta todavia marca estructura real, mientras que la que lo pinta todo no contiene
informacion.

In [ ]:
versus_gt["modo"] = np.where(versus_gt.fg_sam - versus_gt.fg_gt > 0.1, "sobresegmenta",
                     np.where(versus_gt.fg_gt - versus_gt.fg_sam > 0.1, "subsegmenta", "ajustada"))
print(versus_gt.groupby(["video", "modo"]).size().unstack(fill_value=0).to_string(), "\n")

print(versus_gt.groupby("modo").agg(
    n=("iou", "size"), iou=("iou", "mean"),
    precision=("precision", "mean"), recall=("recall", "mean")).round(4).to_string())

print("\n--- cuantas mascaras SAM serian usables ---")
for umbral in (0.5, 0.7):
    ok = versus_gt.iou >= umbral
    print(f"IoU >= {umbral}: {int(ok.sum())}/{len(versus_gt)} ({100 * ok.mean():.0f} %)"
          f" | por video: {versus_gt[ok].video.value_counts().sort_index().to_dict()}")

gana = versus_gt.gain_over_base > 0
print(f"\nsuperan su propia linea base: {int(gana.sum())}/{len(versus_gt)}"
      f" | por video: {versus_gt[gana].video.value_counts().sort_index().to_dict()}")

## 4. Muestra reproducible

Estratificada por video con la semilla fijada en la seccion 1.

In [ ]:
rng = np.random.default_rng(SEED)
chunks = []
for video, k in STRATA.items():
    group = universe[universe.video == video].sort_values("frame_id").reset_index(drop=True)
    take = min(k, len(group))
    chunks.append(group.iloc[np.sort(rng.choice(len(group), size=take, replace=False))])
sample = pd.concat(chunks).reset_index(drop=True)

print(f"semilla {SEED} | estratos {STRATA} | {len(sample)} frames\n")
print(sample[["video", "frame_id", "fg", "components", "has_gt"]].round(4).to_string(index=False))

## 5. Clasificacion visual

Para cada frame de la muestra: frame CLAHE, mascara SAM, overlay y - cuando existe - la mascara
curada al lado. Se mira el overlay y se aplica el criterio de la seccion 1.

In [ ]:
def panel(row):
    fid = row.frame_id
    frame = load_frame(fid)
    mask = load_mask(sam_paths[fid])
    gt = load_mask(curated_paths[fid]) if fid in curated_paths else None
    n_cols = 4 if gt is not None else 3

    fig, axes = plt.subplots(1, n_cols, figsize=(4.6 * n_cols, 4.1))
    if frame is not None:
        axes[0].imshow(frame)
    else:
        axes[0].imshow(np.zeros_like(mask), cmap="gray")
    axes[0].set_title(f"{fid}\nframe CLAHE", fontsize=9)

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title(f"mascara SAM 3 puntos\nfg={mask.mean():.3f}  comp={components(mask)}",
                      fontsize=9)

    if frame is not None:
        overlay = frame.copy()
        overlay[mask] = (0.45 * overlay[mask] + 0.55 * np.array([255, 40, 40])).astype(np.uint8)
        axes[2].imshow(overlay)
    else:
        axes[2].imshow(mask, cmap="gray")
    axes[2].set_title("overlay (rojo = SAM dice estructura)", fontsize=9)

    if gt is not None:
        axes[3].imshow(gt, cmap="gray")
        axes[3].set_title(f"curada (5.1)\nfg={gt.mean():.3f}  "
                          f"IoU={metrics(mask, gt)['iou']:.3f}", fontsize=9)

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


for row in sample.itertuples():
    panel(row)

### 5.1 Tabla de clasificacion

Rellena `CLASIFICACION` con una entrada por `frame_id`: `(clase, nota)`, donde clase es
`aceptable`, `parcial` o `mala`, y nota es el motivo en una linea. Vuelve a ejecutar esta celda y
la de guardado.

In [ ]:
CLASES = {"aceptable", "parcial", "mala"}

if AUDIT_CSV.exists():
    audit = pd.read_csv(AUDIT_CSV).fillna("")
    print(f"cargada la clasificacion existente: {AUDIT_CSV}")
else:
    audit = sample[["video", "frame_id", "fg", "components", "has_gt"]].copy()
    audit["clase"] = ""
    audit["nota"] = ""
    print("plantilla nueva")

# --- EDITA AQUI: una entrada por frame_id ---
CLASIFICACION = {
    # "DeepWater-Video3_frame_0005": ("mala", "ruido disperso, 157 componentes sin relacion con el objeto"),
}

for fid, (clase, nota) in CLASIFICACION.items():
    assert clase in CLASES, f"clase no valida: {clase}"
    assert (audit.frame_id == fid).any(), f"{fid} no esta en la muestra"
    audit.loc[audit.frame_id == fid, ["clase", "nota"]] = [clase, nota]

pendientes = int((audit.clase == "").sum())
print(f"clasificados {len(audit) - pendientes} de {len(audit)} | pendientes {pendientes}")
audit

In [ ]:
audit.to_csv(AUDIT_CSV, index=False)
print("guardado en", AUDIT_CSV)

## 6. Resultado

Porcentaje por categoria, global y por video. La celda falla a proposito si la clasificacion esta
incompleta: un porcentaje sobre una muestra a medio clasificar no significa nada.

In [ ]:
done = audit[audit.clase != ""]
assert len(done) == len(audit), f"faltan {len(audit) - len(done)} frames por clasificar"

orden = ["aceptable", "parcial", "mala"]
dist = (done.clase.value_counts(normalize=True) * 100).reindex(orden).fillna(0)
print("--- distribucion global ---")
for clase, pct in dist.items():
    print(f"{clase:10s} {pct:5.1f} %  ({int((done.clase == clase).sum())} de {len(done)})")

print("\n--- por video ---")
print(pd.crosstab(done.video, done.clase, normalize="index").mul(100).round(1).to_string())

print("\n--- indicadores numericos por categoria ---")
print(done.groupby("clase")[["fg", "components"]].agg(["mean", "min", "max"]).round(3).to_string())

In [ ]:
# 2 o 3 ejemplos por categoria, tomados con la misma semilla
rng_ex = np.random.default_rng(SEED)
for clase in orden:
    group = done[done.clase == clase]
    if group.empty:
        print(f"\nsin ejemplos de {clase}")
        continue
    take = group.iloc[np.sort(rng_ex.choice(len(group), size=min(3, len(group)), replace=False))]
    print(f"\n{'=' * 70}\nejemplos de {clase}\n{'=' * 70}")
    for row in take.itertuples():
        print(f"  {row.frame_id}: {row.nota}")
        panel(row)

## 7. Conclusion y recomendacion

*Cifras de las secciones 3, 3.1 y 3.3, medidas sobre el Drive el 2026-11-19, con las 70 mascaras
curadas de la Task 5.1 ya completa. La clasificacion visual de la seccion 6 se anade cuando este
completa.*

### Lo que muestra la medicion

| | Video1 | Video3 | Video6 | Video7 |
|---|---|---|---|---|
| frames con curado y SAM | 22 | 8 | 8 | 8 |
| estructura real (curada) | 0,741 | 0,780 | **0,081** | 0,891 |
| estructura segun SAM 3 puntos | 0,804 | **0,482** | **0,949** | 0,841 |
| IoU de SAM vs curada | 0,631 | **0,415** | **0,083** | **0,913** |
| precision / recall | 0,765 / 0,821 | **0,868 / 0,483** | **0,083 / 0,980** | 0,987 / 0,925 |
| IoU minimo / maximo | 0,070 / 0,956 | 0,037 / 0,936 | 0,042 / 0,133 | 0,746 / 0,973 |

Solo **28 de las 46 mascaras (61 %) llegan a IoU 0,5**, y 16 (35 %) a 0,7. Las 8 de Video7 pasan
todas; ninguna de Video6 llega a 0,14. Diecinueve superan su propia linea base, pero 7 de esas 19
son de Video6, donde la base es 0,081: superarla ahi no significa nada.

### Interpretacion

1. **Hay dos modos de fallo, no uno.** En Video6 SAM *sobresegmenta* hasta el absurdo: la
   estructura real ocupa el 8 % del frame y la mascara marca el 95 %, con recall 0,98 y precision
   0,083. En Video3 hace lo contrario: *subsegmenta* (0,482 frente a 0,780 real), con precision
   0,868 y recall 0,483 — lo que marca es estructura, pero se deja la mitad fuera.

   Separadas por modo, las 46 se reparten en 23 que sobresegmentan (IoU 0,438, precision 0,461),
   11 que subsegmentan (IoU 0,442, precision 0,994) y **12 ajustadas, que dan IoU 0,852**. Ese
   ultimo grupo es la prueba de que el modelo no es el problema: cuando los tres puntos caen donde
   deben, SAM segmenta bien.
2. **La causa unica explica los dos modos.** En `2_sam_autolabeling_experiment_.ipynb`, si el frame
   no tiene clicks registrados, los "3 puntos" son un placeholder geometrico fijo:
   `[(w//2, h//2), (w//3, h//2), (2*w//3, h//2)]`, tres puntos sobre la linea media horizontal.
   Donde esa linea cae sobre agua, SAM segmenta el agua (Video6). Donde cae sobre un trozo de
   estructura, segmenta ese trozo y nada mas (Video3). No es un fallo de SAM: es un prompt no
   informado, y acierta o falla segun la geometria del video.
3. **Video7 demuestra el punto.** Ahi la estructura ocupa el 89 % del frame, la linea media cae
   sobre ella en todos los frames y el IoU sube a 0,913, el mejor del conjunto. Que las mascaras
   sean buenas en Video7 no dice nada bueno del metodo: dice que el metodo depende de que el objeto
   ocupe casi todo el encuadre.
4. **Video6 es la unica tuberia real del dataset**, el caso mas cercano al alcance original del
   proyecto, y es donde las etiquetas son peores.
5. **La fraccion de estructura no es una propiedad del dataset, es una propiedad de cada video.**
   Va de 0,081 en Video6 a 0,929 en Video2. El sprint se planteo sobre la premisa de que "todo
   estructura" da 0,72, que es aproximadamente el valor de Video1 y de nadie mas: cualquier umbral
   de mejora fijado contra ese 0,72 no se traslada a los demas videos.

### Recomendacion

**No usar las mascaras de `SAM_Masks_Frames_CLAHE` como etiquetas de entrenamiento ni de
evaluacion.** Con el curado completo la recomendacion se puede afinar por video:

- **Video6: descartar sin sustituto.** Ninguna mascara pasa de IoU 0,14.
- **Video3: descartar como etiqueta, conservar como punto de partida.** Lo que marcan es
  estructura (precision 0,868); lo que falta es el resto del objeto, y corregir una mascara que se
  queda corta cuesta menos que una que lo pinta todo.
- **Video1 y Video7: ya hay curado para esos mismos frames**, asi que las SAM no aportan nada.
- `mmseg_pipes` **sigue sin regenerarse**: al 2026-11-19 son las mismas 84/21 imagenes construidas
  desde estas mascaras, con los cuatro videos en los dos splits. Mientras siga asi, cualquier
  entrenamiento que lo use mide el ruido del etiquetado. Afecta a la Task 4.2
  (`3_2_unet_training_diagnosis.ipynb`) y a la Task 5.11. Las 70 curadas ya permiten rehacerlo.
- Si se quiere conservar el etiquetado asistido por SAM, debe hacerse con el cuaderno de la Task
  5.8 (`2_2_assisted_labelling.ipynb`), que toma clicks reales en vez de una posicion fija. Es lo
  que se uso para etiquetar Video7 el 2026-11-10.

### Que queda por cerrar

La clasificacion visual de los 18 frames de la muestra (seccion 5.1), que convierte lo anterior en
los porcentajes por categoria que pide el entregable. La muestra se fijo con el criterio de la
seccion 1 antes de mirar las mascaras y no se cambia ahora que hay mas curado.